In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
    save_features_as_parquet,
)
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    start_profiling,
    stop_profiling,
)
from image_analysis_3D.featurization_utils.texture_utils import measure_3D_texture

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "C4-2"
    patient = "NF0014_T1"
    channel = "DNA"
    compartment = "Cell"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)
channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)

In [3]:
# read in channel mapping
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

In [4]:
start_time, start_mem = start_profiling()

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
    mask_key_name=[channel_n_compartment_mapping[compartment]],
    raw_image_key_name=[channel_n_compartment_mapping[channel]],
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
output_texture_dict = measure_3D_texture(
    object_loader=object_loader,
    distance=3,  # distance in pixels 3 is what CP uses
)
final_df = pd.DataFrame(output_texture_dict)

final_df = final_df.pivot(
    index="object_id",
    columns="texture_name",
    values="texture_value",
)
final_df.reset_index(inplace=True)
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Texture",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)
final_df.insert(0, "image_set", image_set_loader.image_set_name)
final_df.columns.name = None

save_path = save_features_as_parquet(
    parent_path=output_parent_path,
    df=final_df,
    feature_type="Texture",
    channel=channel,
    compartment=compartment,
    cpu_or_gpu=processor_type,
)
final_df.head()

41it [03:26,  5.04s/it]


,image_set,object_id,Cell_DNA_Texture_AngularSecondMoment-256-3,Cell_DNA_Texture_Contrast-256-3,Cell_DNA_Texture_Correlation-256-3,Cell_DNA_Texture_DifferenceEntropy-256-3,Cell_DNA_Texture_DifferenceVariance-256-3,Cell_DNA_Texture_Entropy-256-3,Cell_DNA_Texture_InformationMeasureOfCorrelation1-256-3,Cell_DNA_Texture_InformationMeasureOfCorrelation2-256-3,Cell_DNA_Texture_InverseDifferenceMoment-256-3,Cell_DNA_Texture_SumAverage-256-3,Cell_DNA_Texture_SumEntropy-256-3,Cell_DNA_Texture_SumVariance-256-3,Cell_DNA_Texture_Variance-256-3
0,C4-2,257,0.985003,5.044074,0.893419,0.087232,0.003838,0.135300,-0.599490,0.330502,0.993561,0.690431,0.112061,90.163124,23.801800
1,C4-2,514,0.983615,3.658829,0.878457,0.097047,0.003834,0.143467,-0.606761,0.342528,0.993345,0.543214,0.123689,56.768505,15.106834
2,C4-2,771,0.969134,3.773552,0.862789,0.159313,0.003786,0.243038,-0.620247,0.442931,0.988040,0.798549,0.208653,51.442422,13.803993
3,C4-2,1542,0.992759,6.007845,0.893923,0.051273,0.003864,0.071838,-0.577845,0.237972,0.996703,0.571121,0.060133,107.849022,28.464217
4,C4-2,1799,0.980508,12.000847,0.872517,0.118674,0.003821,0.179034,-0.567857,0.363411,0.991437,1.101892,0.148885,176.175342,47.044047


In [7]:
stop_profiling(
    start_time=start_time,
    start_mem=start_mem,
    feature_type="Texture",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU="CPU",
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Texture_CPU.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: C4-2
        Feature type: Texture
        CPU/GPU: CPU
        Peak memory (tracemalloc): 1862.72 MB
        Current memory (tracemalloc): 299.29 MB
        RSS at end: 476.74 MB
        Time elapsed:
        --- 211.67 seconds ---
        --- 3.53 minutes ---
        --- 0.06 hours ---
    


True